# CH$_4$ - binning

Bin the observational network.

## Imports

In [1]:
from pathlib import Path

import openscm_units
import pandas as pd
import pint
from pydoit_nb.config_handling import get_config_for_step_id

import local.binned_data_interpolation
import local.binning
import local.dependencies
import local.raw_data_processing
from local.config import load_config_from_file

In [2]:
pint.set_application_registry(openscm_units.unit_registry)  # type: ignore

## Define branch this notebook belongs to

In [3]:
step: str = "calculate_ch4_monthly_fifteen_degree_pieces"

## Parameters

In [4]:
config_file: str = "../../dev-config-absolute.yaml"  # config file
step_config_id: str = "only"  # config ID to select for this branch

In [5]:
# Parameters
config_file = "/Users/znicholls/Documents/repos/CMIP-GHG-Concentration-Generation/output-bundles/v1.0.0/v1.0.0-config.yaml"
step_config_id = "only"


## Load config

In [6]:
config = load_config_from_file(Path(config_file))
config_step = get_config_for_step_id(config=config, step=step, step_config_id=step_config_id)

config_process_noaa_surface_flask_data = get_config_for_step_id(
    config=config,
    step="process_noaa_surface_flask_data",
    step_config_id=config_step.gas,
)
config_process_noaa_in_situ_data = get_config_for_step_id(
    config=config,
    step="process_noaa_in_situ_data",
    step_config_id=config_step.gas,
)
config_process_agage_data_gc_md = get_config_for_step_id(
    config=config,
    step="retrieve_and_extract_agage_data",
    step_config_id=f"{config_step.gas}_gc-md_monthly",
)
config_process_ale_data = get_config_for_step_id(
    config=config, step="retrieve_and_extract_ale_data", step_config_id="monthly"
)
config_process_gage_data = get_config_for_step_id(
    config=config, step="retrieve_and_extract_gage_data", step_config_id="monthly"
)

## Action

### Load data

In [7]:
all_data_l = []
for f, dep_short_names in [
    (
        config_process_noaa_surface_flask_data.processed_monthly_data_with_loc_file,
        local.dependencies.load_source_info_short_names(
            config_process_noaa_surface_flask_data.source_info_short_names_file
        ),
    ),
    (
        config_process_noaa_in_situ_data.processed_monthly_data_with_loc_file,
        local.dependencies.load_source_info_short_names(
            config_process_noaa_in_situ_data.source_info_short_names_file
        ),
    ),
    (
        config_process_agage_data_gc_md.processed_monthly_data_with_loc_file,
        local.dependencies.load_source_info_short_names(
            config_process_agage_data_gc_md.source_info_short_names_file
        ),
    ),
    (
        config_process_ale_data.processed_monthly_data_with_loc_file,
        [config_process_ale_data.source_info.short_name],
    ),
    (
        config_process_gage_data.processed_monthly_data_with_loc_file,
        [config_process_gage_data.source_info.short_name],
    ),
]:
    try:
        all_data_l.append(local.raw_data_processing.read_and_check_binning_columns(f))
    except Exception as exc:
        msg = f"Error reading {f}"
        raise ValueError(msg) from exc

    for dsn in dep_short_names:
        local.dependencies.save_dependency_into_db(
            db=config.dependency_db,
            gas=config_step.gas,
            dependency_short_name=dsn,
        )

all_data = pd.concat(all_data_l)
all_data["gas"] = all_data["gas"].str.lower()
all_data = all_data[all_data["gas"] == config_step.gas]
all_data

,gas,reporting_id,year,month,latitude,longitude,value,unit,site_code_filename,site_code,surf_or_ship,source,network,station,measurement_method,time,std. dev.,numb,instrument
0,ch4,month,1983,1,-37.9500,77.5300,1560.970,ppb,ams,AMS,surface,flask,NOAA,ams,flask,NaN,NaN,NaN,NaN
1,ch4,month,1983,1,-75.5600,-27.0200,1561.520,ppb,hba,HBA,surface,flask,NOAA,hba,flask,NaN,NaN,NaN,NaN
2,ch4,month,1983,1,-64.7742,-64.0527,1557.330,ppb,psa,PSA,surface,flask,NOAA,psa,flask,NaN,NaN,NaN,NaN
3,ch4,month,1983,2,-37.9500,77.5300,1559.600,ppb,ams,AMS,surface,flask,NOAA,ams,flask,NaN,NaN,NaN,NaN
4,ch4,month,1983,2,-75.5600,-27.0200,1555.860,ppb,hba,HBA,surface,flask,NOAA,hba,flask,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4103,ch4,NaN,1992,9,-14.2300,-170.5600,1695.664,ppb,NaN,SMO,NaN,GAGE,GAGE,smo,GAGE,NaN,NaN,NaN,NaN
4104,ch4,NaN,1993,9,-14.2300,-170.5600,1700.419,ppb,NaN,SMO,NaN,GAGE,GAGE,smo,GAGE,NaN,NaN,NaN,NaN
4105,ch4,NaN,1994,9,-14.2300,-170.5600,1710.412,ppb,NaN,SMO,NaN,GAGE,GAGE,smo,GAGE,NaN,NaN,NaN,NaN
4106,ch4,NaN,1995,9,-14.2300,-170.5600,1712.275,ppb,NaN,SMO,NaN,GAGE,GAGE,smo,GAGE,NaN,NaN,NaN,NaN


## Bin and average data

- all measurements from a station are first averaged for the month
- then average over all stations
    - stations get equal weight
    - flask/in situ networks (i.e. different measurement methods/techniques)
      are treated as separate stations i.e. get equal weight
- this order is best as you have a better chance of avoiding giving different times more weight by accident
    - properly equally weighting all times in the month would be very hard,
      because you'd need to interpolate to a super fine grid first (one for future research)

In [8]:
all_data_with_bins = local.binning.add_lat_lon_bin_columns(all_data)
all_data_with_bins

,gas,reporting_id,year,month,latitude,longitude,value,unit,site_code_filename,site_code,...,source,network,station,measurement_method,time,std. dev.,numb,instrument,lon_bin,lat_bin
0,ch4,month,1983,1,-37.9500,77.5300,1560.970,ppb,ams,AMS,...,flask,NOAA,ams,flask,NaN,NaN,NaN,NaN,90.0,-37.5
1,ch4,month,1983,1,-75.5600,-27.0200,1561.520,ppb,hba,HBA,...,flask,NOAA,hba,flask,NaN,NaN,NaN,NaN,-30.0,-82.5
2,ch4,month,1983,1,-64.7742,-64.0527,1557.330,ppb,psa,PSA,...,flask,NOAA,psa,flask,NaN,NaN,NaN,NaN,-90.0,-67.5
3,ch4,month,1983,2,-37.9500,77.5300,1559.600,ppb,ams,AMS,...,flask,NOAA,ams,flask,NaN,NaN,NaN,NaN,90.0,-37.5
4,ch4,month,1983,2,-75.5600,-27.0200,1555.860,ppb,hba,HBA,...,flask,NOAA,hba,flask,NaN,NaN,NaN,NaN,-30.0,-82.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4103,ch4,NaN,1992,9,-14.2300,-170.5600,1695.664,ppb,NaN,SMO,...,GAGE,GAGE,smo,GAGE,NaN,NaN,NaN,NaN,-150.0,-7.5
4104,ch4,NaN,1993,9,-14.2300,-170.5600,1700.419,ppb,NaN,SMO,...,GAGE,GAGE,smo,GAGE,NaN,NaN,NaN,NaN,-150.0,-7.5
4105,ch4,NaN,1994,9,-14.2300,-170.5600,1710.412,ppb,NaN,SMO,...,GAGE,GAGE,smo,GAGE,NaN,NaN,NaN,NaN,-150.0,-7.5
4106,ch4,NaN,1995,9,-14.2300,-170.5600,1712.275,ppb,NaN,SMO,...,GAGE,GAGE,smo,GAGE,NaN,NaN,NaN,NaN,-150.0,-7.5


In [9]:
print(local.binning.get_network_summary(all_data_with_bins))

Collating data from:
- AGAGE gc-md (5 stations: cgo, mhd, rpb, smo, thd)
- GAGE GAGE (4 stations: cgo, mhd, org, smo)
- NOAA flask (95 stations: 000, abp, alt, ams ... wis, wkt, wlg, zep)
- NOAA insitu (3 stations: brw, mko, mlo)


In [10]:
bin_averages = local.binning.calculate_bin_averages(all_data_with_bins)
bin_averages

Will ignore columns: ['reporting_id', 'latitude', 'longitude', 'site_code_filename', 'site_code', 'surf_or_ship', 'source', 'time', 'std. dev.', 'numb', 'instrument']
Took mean over ['index']
Took mean over ['measurement_method', 'network', 'station']


,gas,unit,year,month,lat_bin,lon_bin,value
0,ch4,ppb,1983,1,-82.5,-30.0,1561.520
1,ch4,ppb,1983,1,-67.5,-90.0,1557.330
2,ch4,ppb,1983,1,-37.5,90.0,1560.970
3,ch4,ppb,1983,2,-82.5,-30.0,1555.810
4,ch4,ppb,1983,2,-67.5,-90.0,1549.560
...,...,...,...,...,...,...,...
15434,ch4,ppb,2024,5,37.5,-150.0,2002.509
15435,ch4,ppb,2024,5,52.5,-30.0,2004.661
15436,ch4,ppb,2024,6,-37.5,150.0,1880.897
15437,ch4,ppb,2024,6,37.5,-150.0,1989.913


### Save

In [11]:
local.binned_data_interpolation.check_data_columns_for_binned_data_interpolation(bin_averages)
assert set(bin_averages["gas"]) == {config_step.gas}

In [12]:
config_step.processed_bin_averages_file.parent.mkdir(exist_ok=True, parents=True)
bin_averages.to_csv(config_step.processed_bin_averages_file, index=False)
bin_averages

,gas,unit,year,month,lat_bin,lon_bin,value
0,ch4,ppb,1983,1,-82.5,-30.0,1561.520
1,ch4,ppb,1983,1,-67.5,-90.0,1557.330
2,ch4,ppb,1983,1,-37.5,90.0,1560.970
3,ch4,ppb,1983,2,-82.5,-30.0,1555.810
4,ch4,ppb,1983,2,-67.5,-90.0,1549.560
...,...,...,...,...,...,...,...
15434,ch4,ppb,2024,5,37.5,-150.0,2002.509
15435,ch4,ppb,2024,5,52.5,-30.0,2004.661
15436,ch4,ppb,2024,6,-37.5,150.0,1880.897
15437,ch4,ppb,2024,6,37.5,-150.0,1989.913
